# 02a — RetinaFace Preprocessing

This notebook creates and verifies the cropped-face dataset used by **E1.1** and **E2.2**.


## 1. Controls

The default settings perform the complete build and audit. Existing face crops with matching filenames are skipped, allowing a disconnected run to resume safely.

There is deliberately no force-delete option in this notebook.

In [1]:
INSTALL_DEPENDENCIES = True
RUN_FACE_DATASET_BUILD = True
RUN_FACE_DATASET_AUDIT = True
RUN_FACE_DATASET_VISUAL_CHECK = False

FACE_MARGIN_RATIO = 0.15
FACE_SIZE = 224
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

assert 0.0 <= FACE_MARGIN_RATIO <= 1.0
assert FACE_SIZE > 0

print("Install RetinaFace dependencies:", INSTALL_DEPENDENCIES)
print("Build face_dataset:", RUN_FACE_DATASET_BUILD)
print("Audit face_dataset:", RUN_FACE_DATASET_AUDIT)
print("Face margin:", FACE_MARGIN_RATIO)
print("Saved face size:", f"{FACE_SIZE}x{FACE_SIZE}")

Install RetinaFace dependencies: True
Build face_dataset: True
Audit face_dataset: True
Face margin: 0.15
Saved face size: 224x224


## 2. Mount Drive and load dependencies

In [2]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

if INSTALL_DEPENDENCIES:
    import subprocess
    import sys
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "retina-face", "tf-keras",
    ])

import random
import time
from pathlib import Path

import cv2
import pandas as pd
from retinaface import RetinaFace

BASE_PATH = Path("/content/drive/MyDrive/deepfake_project")
DATASET_ROOT = BASE_PATH / "dataset"
FACE_DATASET_ROOT = BASE_PATH / "face_dataset"
RESULTS_ROOT = BASE_PATH / "results"
SPLIT_MANIFEST_PATH = BASE_PATH / "ffpp_video_split_manifest.json"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

if not SPLIT_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "The video-level split manifest is missing. Run the combined project "
        "setup and FF++ data-preparation notebook first."
    )

print("Input dataset:", DATASET_ROOT)
print("Output face dataset:", FACE_DATASET_ROOT)
print("Split manifest:", SPLIT_MANIFEST_PATH)

Mounted at /content/drive
Input dataset: /content/drive/MyDrive/deepfake_project/dataset
Output face dataset: /content/drive/MyDrive/deepfake_project/face_dataset
Split manifest: /content/drive/MyDrive/deepfake_project/ffpp_video_split_manifest.json


## 3. Verify the full-frame input dataset

This check runs before face detection. All six folders must exist and contain images. The expected clean split contains 1,400 training and 300 validation/test frames per class.

In [3]:
def list_images(folder):
    if not folder.is_dir():
        return []
    return sorted(
        path for path in folder.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


input_rows = []
input_images = {}
for split in ("train", "val", "test"):
    for class_name in ("real", "fake"):
        folder = DATASET_ROOT / split / class_name
        images = list_images(folder)
        input_images[(split, class_name)] = images
        input_rows.append({
            "split": split,
            "class": class_name,
            "input_frames": len(images),
            "folder_exists": folder.is_dir(),
        })

input_table = pd.DataFrame(input_rows)
display(input_table.sort_values(["split", "class"]))

if not input_table["folder_exists"].all():
    raise FileNotFoundError("One or more full-frame dataset folders are missing.")
if (input_table["input_frames"] == 0).any():
    raise RuntimeError("One or more full-frame dataset folders are empty.")

print("Full-frame input dataset is ready.")

,split,class,input_frames,folder_exists
5,test,fake,300,True
4,test,real,300,True
1,train,fake,1400,True
0,train,real,1400,True
3,val,fake,300,True
2,val,real,300,True


Full-frame input dataset is ready.


## 4. RetinaFace crop definition

For each input image, the largest detected face is selected. The bounding box is expanded by 15% on each side, clipped to the image boundaries and resized to 224 × 224 pixels.

In [4]:
def crop_largest_face(image):
    # RGB input, while the crop saved by OpenCV remains in BGR order.
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    try:
        detections = RetinaFace.detect_faces(image_rgb)
    except Exception:
        return None
    if not isinstance(detections, dict):
        return None

    best_box = None
    best_area = 0
    for detection in detections.values():
        x1, y1, x2, y2 = detection["facial_area"]
        area = max(0, x2 - x1) * max(0, y2 - y1)
        if area > best_area:
            best_area = area
            best_box = (x1, y1, x2, y2)

    if best_box is None:
        return None

    x1, y1, x2, y2 = best_box
    height, width = image.shape[:2]
    face_width = x2 - x1
    face_height = y2 - y1
    margin_x = int(face_width * FACE_MARGIN_RATIO)
    margin_y = int(face_height * FACE_MARGIN_RATIO)

    x1 = max(int(x1 - margin_x), 0)
    y1 = max(int(y1 - margin_y), 0)
    x2 = min(int(x2 + margin_x), width)
    y2 = min(int(y2 + margin_y), height)

    crop = image[y1:y2, x1:x2]
    if crop.size == 0:
        return None

    return cv2.resize(crop, (FACE_SIZE, FACE_SIZE), interpolation=cv2.INTER_AREA)

## 5. Build or safely resume `face_dataset`

Each face crop retains the filename and train, validation or test folder of its corresponding full frame. Existing crops are skipped so that an interrupted run can continue without rebuilding completed files. Images for which a valid face cannot be detected are recorded in the build log.

In [5]:
if RUN_FACE_DATASET_BUILD:
    build_rows = []
    start_time = time.time()

    for split in ("train", "val", "test"):
        for class_name in ("real", "fake"):
            output_folder = FACE_DATASET_ROOT / split / class_name
            output_folder.mkdir(parents=True, exist_ok=True)
            images = input_images[(split, class_name)]

            print(f"Processing {split}/{class_name}: {len(images)} frames")

            for position, input_path in enumerate(images, start=1):
                output_path = output_folder / input_path.name

                if output_path.exists():
                    status = "already existed"
                else:
                    image = cv2.imread(str(input_path))
                    if image is None:
                        status = "unreadable input"
                    else:
                        face_crop = crop_largest_face(image)
                        if face_crop is None:
                            status = "no face detected"
                        else:
                            saved = cv2.imwrite(str(output_path), face_crop)
                            status = "saved" if saved else "write failed"

                build_rows.append({
                    "split": split,
                    "class": class_name,
                    "input_filename": input_path.name,
                    "output_filename": output_path.name,
                    "status": status,
                })

                if position % 100 == 0 or position == len(images):
                    print(f"  {position}/{len(images)}")

    face_build_log = pd.DataFrame(build_rows)
    log_path = RESULTS_ROOT / "face_dataset_build_log.csv"
    face_build_log.to_csv(log_path, index=False)

    print(f"Face preprocessing completed in {(time.time() - start_time) / 60:.2f} minutes.")
    display(
        face_build_log.groupby(["split", "class", "status"])
        .size()
        .rename("images")
        .reset_index()
    )
    print("Build log:", log_path)
else:
    print("Face-dataset build skipped.")

Processing train/real: 1400 frames
26-09-12 03:24:18 - Directory /root/.deepface created
26-09-12 03:24:18 - Directory /root/.deepface/weights created
26-09-12 03:24:18 - retinaface.h5 will be downloaded from the url https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5


Downloading...
From: https://github.com/serengil/deepface_models/releases/download/v1.0/retinaface.h5
To: /root/.deepface/weights/retinaface.h5
100%|██████████| 119M/119M [00:00<00:00, 150MB/s]


  100/1400
  200/1400
  300/1400
  400/1400
  500/1400
  600/1400
  700/1400
  800/1400
  900/1400
  1000/1400
  1100/1400
  1200/1400
  1300/1400
  1400/1400
Processing train/fake: 1400 frames
  100/1400
  200/1400
  300/1400
  400/1400
  500/1400
  600/1400
  700/1400
  800/1400
  900/1400
  1000/1400
  1100/1400
  1200/1400
  1300/1400
  1400/1400
Processing val/real: 300 frames
  100/300
  200/300
  300/300
Processing val/fake: 300 frames
  100/300
  200/300
  300/300
Processing test/real: 300 frames
  100/300
  200/300
  300/300
Processing test/fake: 300 frames
  100/300
  200/300
  300/300
Face preprocessing completed in 38.97 minutes.


,split,class,status,images
0,test,fake,no face detected,1
1,test,fake,saved,299
2,test,real,no face detected,8
3,test,real,saved,292
4,train,fake,already existed,984
5,train,fake,no face detected,2
6,train,fake,saved,414
7,train,real,already existed,1342
8,train,real,no face detected,58
9,val,fake,saved,300


Build log: /content/drive/MyDrive/deepfake_project/results/face_dataset_build_log.csv


## 6. Face-dataset audit

The audit compares the face crops with the exact full-frame inputs, checks filename preservation and tests for source-video overlap across splits.

In [6]:
def recover_video_id(filename):
    stem = Path(filename).stem
    if "_frame_" not in stem:
        return stem
    return stem.rsplit("_frame_", 1)[0]


if RUN_FACE_DATASET_AUDIT:
    audit_rows = []
    face_video_ids = {}

    for split in ("train", "val", "test"):
        for class_name in ("real", "fake"):
            input_names = {
                path.name for path in input_images[(split, class_name)]
            }
            output_folder = FACE_DATASET_ROOT / split / class_name
            output_paths = list_images(output_folder)
            output_names = {path.name for path in output_paths}
            unexpected_names = output_names - input_names
            missing_names = input_names - output_names
            video_ids = {recover_video_id(name) for name in output_names}
            face_video_ids[(split, class_name)] = video_ids

            audit_rows.append({
                "split": split,
                "class": class_name,
                "input_frames": len(input_names),
                "face_crops": len(output_names),
                "detection_rate_pct": round(100 * len(output_names) / len(input_names), 2),
                "source_videos": len(video_ids),
                "missing_crops": len(missing_names),
                "unexpected_files": len(unexpected_names),
            })

    face_audit_table = pd.DataFrame(audit_rows)
    display(face_audit_table.sort_values(["split", "class"]))

    if (face_audit_table["face_crops"] == 0).any():
        raise RuntimeError("At least one face-dataset folder is empty.")
    if (face_audit_table["unexpected_files"] > 0).any():
        raise RuntimeError(
            "Unexpected face files were found that do not match the full-frame input dataset."
        )

    overlap_rows = []
    for class_name in ("real", "fake"):
        for first, second in (("train", "val"), ("train", "test"), ("val", "test")):
            overlap = (
                face_video_ids[(first, class_name)]
                & face_video_ids[(second, class_name)]
            )
            overlap_rows.append({
                "class": class_name,
                "comparison": f"{first} vs {second}",
                "overlapping_videos": len(overlap),
                "examples": sorted(overlap)[:5],
            })

    face_overlap_table = pd.DataFrame(overlap_rows)
    display(face_overlap_table)

    if (face_overlap_table["overlapping_videos"] > 0).any():
        raise RuntimeError("Source-video overlap was detected in face_dataset.")

    print("PASS: face_dataset preserves the video-level split with no overlap.")
else:
    print("Face-dataset audit skipped.")

,split,class,input_frames,face_crops,detection_rate_pct,source_videos,missing_crops,unexpected_files
5,test,fake,300,299,99.67,30,1,0
4,test,real,300,292,97.33,30,8,0
1,train,fake,1400,1398,99.86,140,2,0
0,train,real,1400,1342,95.86,140,58,0
3,val,fake,300,300,100.00,30,0,0
2,val,real,300,290,96.67,30,10,0


,class,comparison,overlapping_videos,examples
0,real,train vs val,0,[]
1,real,train vs test,0,[]
2,real,val vs test,0,[]
3,fake,train vs val,0,[]
4,fake,train vs test,0,[]
5,fake,val vs test,0,[]


PASS: face_dataset preserves the video-level split with no overlap.


## 7. Optional visual check

This displays random training face crops without modifying the dataset.

In [7]:
if RUN_FACE_DATASET_VISUAL_CHECK:
    import matplotlib.pyplot as plt
    from PIL import Image

    sample_pool = (
        list_images(FACE_DATASET_ROOT / "train" / "real")
        + list_images(FACE_DATASET_ROOT / "train" / "fake")
    )
    if not sample_pool:
        print("No face crops are available for visual inspection.")
    else:
        samples = random.sample(sample_pool, min(9, len(sample_pool)))
        plt.figure(figsize=(10, 10))
        for index, image_path in enumerate(samples, start=1):
            image = Image.open(image_path).convert("RGB")
            plt.subplot(3, 3, index)
            plt.imshow(image)
            plt.title(image_path.parent.name)
            plt.axis("off")
        plt.tight_layout()
        plt.show()
else:
    print("Face-dataset visual check skipped.")

Face-dataset visual check skipped.


## 8. Downstream notebooks

After the audit passes, the prepared face crops are used by:

- `03b_E11_Face_CNN_Training.ipynb`
- `03e_E22_Face_EfficientNet_Training.ipynb`
- the corresponding `04` FaceForensics++ evaluation notebooks

